In [1]:
# ============================================================
# 10_intermediate_report_result_collector.ipynb
# BUILD ONE COMPLETE RESULT HUB FOR INTERMEDIATE REPORT
# ============================================================

import json
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_colwidth", 300)

# ============================================================
# 1. PATHS
# ============================================================
cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

SPLIT_DIR = ROOT / "data" / "processed" / "splits"
MANIFEST_DIR = ROOT / "data" / "processed" / "manifests"
RESULTS_DIR = ROOT / "results"
OUTPUT_DIR = ROOT / "results" / "intermediate_report_bundle"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALL_PROTOCOLS_SUMMARY = SPLIT_DIR / "all_protocols_summary.csv"
FRAME_SUMMARY_CSV = MANIFEST_DIR / "frame_extraction_trimmed_summary.csv"

print("ROOT:", ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

# ============================================================
# 2. COLLECT ALL metrics.json FILES
# ============================================================
metrics_files = sorted(RESULTS_DIR.rglob("metrics.json"))
print("\nFound metrics.json files:", len(metrics_files))
for p in metrics_files[:20]:
    print(" ", p)

if len(metrics_files) == 0:
    raise ValueError("No metrics.json files found inside results/")

# ============================================================
# 3. LOAD ALL METRICS INTO MASTER TABLE
# ============================================================
rows = []

for mf in metrics_files:
    try:
        with open(mf, "r") as f:
            m = json.load(f)
    except Exception as e:
        print("Skipping unreadable:", mf, "|", e)
        continue

    row = {
        "metrics_path": str(mf),
        "run_dir": str(mf.parent),
        "protocol_name": m.get("protocol_name"),
        "frame_source": m.get("frame_source"),
        "model_name": m.get("model_name"),
        "num_sample_frames": m.get("num_sample_frames"),
        "batch_size": m.get("batch_size"),
        "best_epoch": m.get("best_epoch"),
        "best_val_macro_f1": m.get("best_val_macro_f1"),
        "test_accuracy": m.get("test_accuracy"),
        "test_macro_f1": m.get("test_macro_f1"),
        "test_weighted_f1": m.get("test_weighted_f1"),
        "train_time_minutes": m.get("train_time_minutes"),
        "inference_time_seconds": m.get("inference_time_seconds"),
        "videos_per_second": m.get("videos_per_second"),
        "train_rows": m.get("train_rows"),
        "val_rows": m.get("val_rows"),
        "test_rows": m.get("test_rows"),
        "predictions_csv": m.get("predictions_csv"),
        "classification_report_csv": m.get("classification_report_csv"),
        "confusion_matrix_png": m.get("confusion_matrix_png"),
        "checkpoint_best": m.get("checkpoint_best"),
    }

    per_class = m.get("per_class_f1", {})
    if isinstance(per_class, dict):
        for cls_name, score in per_class.items():
            row[f"f1_{cls_name}"] = score

    # fallback parsing from folder name if any field missing
    if row["protocol_name"] is None or row["frame_source"] is None or row["model_name"] is None:
        parts = mf.parent.name.split("__")
        if len(parts) >= 3:
            row["protocol_name"] = row["protocol_name"] or parts[0]
            row["frame_source"] = row["frame_source"] or parts[1]
            row["model_name"] = row["model_name"] or parts[2]

    rows.append(row)

master_df = pd.DataFrame(rows)

if len(master_df) == 0:
    raise ValueError("Metrics collection failed. No rows loaded.")

# numeric cleanup
numeric_cols = [
    "num_sample_frames", "batch_size", "best_epoch",
    "best_val_macro_f1", "test_accuracy", "test_macro_f1", "test_weighted_f1",
    "train_time_minutes", "inference_time_seconds", "videos_per_second",
    "train_rows", "val_rows", "test_rows"
]
for c in numeric_cols:
    if c in master_df.columns:
        master_df[c] = pd.to_numeric(master_df[c], errors="coerce")

# remove duplicates if same experiment saved twice
master_df = master_df.drop_duplicates(
    subset=["protocol_name", "frame_source", "model_name", "metrics_path"],
    keep="last"
).reset_index(drop=True)

master_df = master_df.sort_values(
    by=["protocol_name", "frame_source", "test_macro_f1"],
    ascending=[True, True, False]
).reset_index(drop=True)

MASTER_CSV = OUTPUT_DIR / "master_experiment_results.csv"
master_df.to_csv(MASTER_CSV, index=False)

print("\nSaved MASTER_CSV:", MASTER_CSV)
print("Master shape:", master_df.shape)

# ============================================================
# 4. LOAD PROTOCOL COVERAGE SUMMARY IF AVAILABLE
# ============================================================
if ALL_PROTOCOLS_SUMMARY.exists():
    protocol_df = pd.read_csv(ALL_PROTOCOLS_SUMMARY)
else:
    protocol_df = pd.DataFrame()

PROTOCOL_COVERAGE_CSV = OUTPUT_DIR / "protocol_coverage_summary.csv"
if len(protocol_df) > 0:
    protocol_df.to_csv(PROTOCOL_COVERAGE_CSV, index=False)
    print("Saved PROTOCOL_COVERAGE_CSV:", PROTOCOL_COVERAGE_CSV)

# ============================================================
# 5. BUILD PROTOCOL COMPLETION TABLE
# ============================================================
protocol_completion = (
    master_df.groupby(["protocol_name", "frame_source"])["model_name"]
    .nunique()
    .reset_index(name="num_models_finished")
)

protocol_completion["is_complete_for_3_models"] = protocol_completion["num_models_finished"] >= 3

PROTOCOL_COMPLETION_CSV = OUTPUT_DIR / "protocol_completion_table.csv"
protocol_completion.to_csv(PROTOCOL_COMPLETION_CSV, index=False)

# ============================================================
# 6. BEST MODEL PER PROTOCOL
# ============================================================
best_idx = master_df.groupby(["protocol_name", "frame_source"])["test_macro_f1"].idxmax()
best_df = master_df.loc[best_idx].copy().reset_index(drop=True)

BEST_CSV = OUTPUT_DIR / "best_model_per_protocol.csv"
best_df.to_csv(BEST_CSV, index=False)

# ============================================================
# 7. AVERAGE PERFORMANCE TABLES
# ============================================================
avg_by_model = (
    master_df.groupby("model_name")[[
        "test_accuracy", "test_macro_f1", "test_weighted_f1",
        "train_time_minutes", "videos_per_second"
    ]]
    .mean(numeric_only=True)
    .round(4)
    .reset_index()
    .sort_values("test_macro_f1", ascending=False)
)

AVG_MODEL_CSV = OUTPUT_DIR / "average_by_model.csv"
avg_by_model.to_csv(AVG_MODEL_CSV, index=False)

avg_by_source = (
    master_df.groupby("frame_source")[[
        "test_accuracy", "test_macro_f1", "test_weighted_f1",
        "train_time_minutes", "videos_per_second"
    ]]
    .mean(numeric_only=True)
    .round(4)
    .reset_index()
    .sort_values("test_macro_f1", ascending=False)
)

AVG_SOURCE_CSV = OUTPUT_DIR / "average_by_frame_source.csv"
avg_by_source.to_csv(AVG_SOURCE_CSV, index=False)

# ============================================================
# 8. LONG PER-CLASS F1 TABLE
# ============================================================
class_names = ["doctor", "emergency", "fire", "help", "robbery", "sit_down", "stand_up"]
per_class_rows = []

for _, row in master_df.iterrows():
    for cls in class_names:
        col = f"f1_{cls}"
        if col in master_df.columns and pd.notna(row.get(col, np.nan)):
            per_class_rows.append({
                "protocol_name": row["protocol_name"],
                "frame_source": row["frame_source"],
                "model_name": row["model_name"],
                "class_name": cls,
                "f1_score": row[col],
                "test_accuracy": row["test_accuracy"],
                "test_macro_f1": row["test_macro_f1"],
            })

per_class_df = pd.DataFrame(per_class_rows)
PER_CLASS_CSV = OUTPUT_DIR / "per_class_f1_long.csv"
if len(per_class_df) > 0:
    per_class_df.to_csv(PER_CLASS_CSV, index=False)

# ============================================================
# 9. PROTOCOL-WISE PIVOT TABLES
# ============================================================
pivot_macro = master_df.pivot_table(
    index=["protocol_name", "frame_source"],
    columns="model_name",
    values="test_macro_f1",
    aggfunc="max"
).reset_index()

pivot_acc = master_df.pivot_table(
    index=["protocol_name", "frame_source"],
    columns="model_name",
    values="test_accuracy",
    aggfunc="max"
).reset_index()

PIVOT_MACRO_CSV = OUTPUT_DIR / "pivot_test_macro_f1.csv"
PIVOT_ACC_CSV = OUTPUT_DIR / "pivot_test_accuracy.csv"
pivot_macro.to_csv(PIVOT_MACRO_CSV, index=False)
pivot_acc.to_csv(PIVOT_ACC_CSV, index=False)

# ============================================================
# 10. RUNTIME SUMMARY
# ============================================================
runtime_summary = {
    "num_total_experiments_collected": int(len(master_df)),
    "num_unique_protocols": int(master_df["protocol_name"].nunique()),
    "num_unique_frame_sources": int(master_df["frame_source"].nunique()),
    "num_unique_models": int(master_df["model_name"].nunique()),
    "total_train_time_minutes_sum": float(master_df["train_time_minutes"].sum(skipna=True)),
    "total_train_time_hours_sum": float(master_df["train_time_minutes"].sum(skipna=True) / 60.0),
    "average_train_time_minutes": float(master_df["train_time_minutes"].mean(skipna=True)),
    "fastest_model_avg_videos_per_second": (
        avg_by_model.sort_values("videos_per_second", ascending=False).iloc[0]["model_name"]
        if len(avg_by_model) > 0 else None
    ),
    "best_model_avg_macro_f1": (
        avg_by_model.sort_values("test_macro_f1", ascending=False).iloc[0]["model_name"]
        if len(avg_by_model) > 0 else None
    )
}

RUNTIME_JSON = OUTPUT_DIR / "runtime_summary.json"
with open(RUNTIME_JSON, "w") as f:
    json.dump(runtime_summary, f, indent=4)

# ============================================================
# 11. BUILD A MARKDOWN SUMMARY FILE FOR REPORT WRITING
# ============================================================
report_md = []

report_md.append("# Intermediate Report Result Summary\n")
report_md.append("## 1. Project Snapshot\n")
report_md.append(f"- Total collected experiments: **{len(master_df)}**\n")
report_md.append(f"- Unique protocols: **{master_df['protocol_name'].nunique()}**\n")
report_md.append(f"- Unique frame sources: **{master_df['frame_source'].nunique()}**\n")
report_md.append(f"- Unique models: **{master_df['model_name'].nunique()}**\n")
report_md.append(f"- Total training time (sum): **{runtime_summary['total_train_time_hours_sum']:.2f} hours**\n")

report_md.append("\n## 2. Current Experimental Dimensions\n")
report_md.append("- Frame sources used: RGB, RGB_BGREM\n")
report_md.append("- Models used: resnet18, efficientnet_b0, mobilenet_v3_small\n")
report_md.append("- Evaluation includes Protocol A and multiple cross-distance protocols where available.\n")

report_md.append("\n## 3. Best Model per Protocol and Frame Source\n")
for _, row in best_df.sort_values(["protocol_name", "frame_source"]).iterrows():
    report_md.append(
        f"- **{row['protocol_name']} | {row['frame_source']}** → "
        f"{row['model_name']} "
        f"(test_macro_f1={row['test_macro_f1']:.4f}, "
        f"test_accuracy={row['test_accuracy']:.4f})"
    )

report_md.append("\n## 4. Average Performance by Model\n")
for _, row in avg_by_model.iterrows():
    report_md.append(
        f"- **{row['model_name']}** → "
        f"avg test_macro_f1={row['test_macro_f1']:.4f}, "
        f"avg test_accuracy={row['test_accuracy']:.4f}, "
        f"avg train_time_minutes={row['train_time_minutes']:.2f}"
    )

report_md.append("\n## 5. Average Performance by Frame Source\n")
for _, row in avg_by_source.iterrows():
    report_md.append(
        f"- **{row['frame_source']}** → "
        f"avg test_macro_f1={row['test_macro_f1']:.4f}, "
        f"avg test_accuracy={row['test_accuracy']:.4f}"
    )

report_md.append("\n## 6. Files Generated\n")
report_md.append(f"- Master results CSV: `{MASTER_CSV}`")
report_md.append(f"- Best-per-protocol CSV: `{BEST_CSV}`")
report_md.append(f"- Average-by-model CSV: `{AVG_MODEL_CSV}`")
report_md.append(f"- Average-by-frame-source CSV: `{AVG_SOURCE_CSV}`")
report_md.append(f"- Per-class F1 CSV: `{PER_CLASS_CSV}`")
report_md.append(f"- Runtime summary JSON: `{RUNTIME_JSON}`")
report_md.append(f"- Protocol completion CSV: `{PROTOCOL_COMPLETION_CSV}`")

REPORT_MD = OUTPUT_DIR / "intermediate_report_summary.md"
REPORT_MD.write_text("\n".join(report_md), encoding="utf-8")

print("Saved REPORT_MD:", REPORT_MD)

# ============================================================
# 12. DISPLAY INSIGHT TABLES
# ============================================================
print("\n========== MASTER RESULTS TABLE ==========")
display(master_df[[
    "protocol_name", "frame_source", "model_name",
    "test_accuracy", "test_macro_f1", "test_weighted_f1",
    "train_time_minutes", "videos_per_second"
]].sort_values(["protocol_name", "frame_source", "test_macro_f1"], ascending=[True, True, False]))

print("\n========== BEST MODEL PER PROTOCOL ==========")
display(best_df[[
    "protocol_name", "frame_source", "model_name",
    "test_accuracy", "test_macro_f1", "test_weighted_f1",
    "train_time_minutes", "videos_per_second"
]].sort_values(["protocol_name", "frame_source"]))

print("\n========== AVERAGE BY MODEL ==========")
display(avg_by_model)

print("\n========== AVERAGE BY FRAME SOURCE ==========")
display(avg_by_source)

print("\n========== PROTOCOL COMPLETION ==========")
display(protocol_completion.sort_values(["protocol_name", "frame_source"]))

if len(per_class_df) > 0:
    print("\n========== PER-CLASS F1 (LONG FORMAT) ==========")
    display(per_class_df.head(50))

print("\n========== RUNTIME SUMMARY ==========")
print(json.dumps(runtime_summary, indent=4))

print("\nAll summary files saved in:", OUTPUT_DIR)

ROOT: /data/Sajjan_Singh/gesture_recognition
RESULTS_DIR: /data/Sajjan_Singh/gesture_recognition/results
OUTPUT_DIR: /data/Sajjan_Singh/gesture_recognition/results/intermediate_report_bundle

Found metrics.json files: 80
  /data/Sajjan_Singh/gesture_recognition/results/rgb_framevote_efficientnet_b0_f8_img224/metrics.json
  /data/Sajjan_Singh/gesture_recognition/results/rgb_framevote_mobilenet_v3_small_f8_img224/metrics.json
  /data/Sajjan_Singh/gesture_recognition/results/rgb_framevote_resnet18_f8_img224/metrics.json
  /data/Sajjan_Singh/gesture_recognition/results/rgb_trimmed_rgb_efficientnet_b0_f8_img224/metrics.json
  /data/Sajjan_Singh/gesture_recognition/results/rgb_trimmed_rgb_mobilenet_v3_small_f8_img224/metrics.json
  /data/Sajjan_Singh/gesture_recognition/results/rgb_trimmed_rgb_resnet18_f8_img224/metrics.json
  /data/Sajjan_Singh/gesture_recognition/results/stage5_remaining_protocols/protocol_B_4_6_to_4__rgb__efficientnet_b0__f32__img224/metrics.json
  /data/Sajjan_Singh/gest

,protocol_name,frame_source,model_name,test_accuracy,test_macro_f1,test_weighted_f1,train_time_minutes,videos_per_second
0,protocol_A,RGB,efficientnet_b0,0.884354,0.885118,0.885118,133.153143,1.139901
1,protocol_A,RGB,efficientnet_b0,0.884354,0.885118,0.885118,135.243664,0.972483
2,protocol_A,RGB,resnet18,0.809524,0.807739,0.807739,133.159111,0.930557
3,protocol_A,RGB,resnet18,0.809524,0.807739,0.807739,148.005852,0.456316
4,protocol_A,RGB,mobilenet_v3_small,0.755102,0.757593,0.757593,130.139361,1.146739
...,...,...,...,...,...,...,...,...
75,None,RGB,resnet18,0.816327,0.811662,NaN,NaN,NaN
76,None,RGB,mobilenet_v3_small,0.687075,0.679070,NaN,NaN,NaN
77,None,None,efficientnet_b0,0.156463,0.060989,NaN,NaN,NaN
78,None,None,resnet18,0.156463,0.060989,NaN,NaN,NaN



========== BEST MODEL PER PROTOCOL ==========


,protocol_name,frame_source,model_name,test_accuracy,test_macro_f1,test_weighted_f1,train_time_minutes,videos_per_second
0,protocol_A,RGB,efficientnet_b0,0.884354,0.885118,0.885118,133.153143,1.139901
1,protocol_A,RGB_BGREM,resnet18,0.829932,0.828292,0.828292,87.946307,0.471911
2,protocol_B_4_6_to_4,RGB,efficientnet_b0,0.970588,0.970560,0.970560,119.939490,0.235313
3,protocol_B_4_6_to_8,RGB,efficientnet_b0,0.627820,0.628604,0.628604,63.282597,0.695280
4,protocol_B_4_6_to_8,RGB_BGREM,efficientnet_b0,0.624060,0.624181,0.624181,98.401112,0.410089
5,protocol_B_4_8_to_6,RGB,resnet18,0.808163,0.800273,0.800273,110.937489,0.757005
6,protocol_B_4_8_to_6,RGB_BGREM,resnet18,0.812245,0.811072,0.811072,100.707012,0.899594
7,protocol_B_4_to_4,RGB,efficientnet_b0,0.949580,0.949704,0.949704,23.688075,2.373255
8,protocol_B_4_to_6,RGB,resnet18,0.555102,0.557025,0.557025,15.180778,0.405149
9,protocol_B_4_to_8,RGB,efficientnet_b0,0.428571,0.401573,0.401573,21.344228,0.631743



========== AVERAGE BY MODEL ==========


...


========== PROTOCOL COMPLETION ==========


,protocol_name,frame_source,num_models_finished,is_complete_for_3_models
0,protocol_A,RGB,3,True
1,protocol_A,RGB_BGREM,3,True
2,protocol_B_4_6_to_4,RGB,3,True
3,protocol_B_4_6_to_8,RGB,3,True
4,protocol_B_4_6_to_8,RGB_BGREM,3,True
5,protocol_B_4_8_to_6,RGB,3,True
6,protocol_B_4_8_to_6,RGB_BGREM,3,True
7,protocol_B_4_to_4,RGB,3,True
8,protocol_B_4_to_6,RGB,3,True
9,protocol_B_4_to_8,RGB,3,True



========== PER-CLASS F1 (LONG FORMAT) ==========


,protocol_name,frame_source,model_name,class_name,f1_score,test_accuracy,test_macro_f1
0,protocol_A,RGB,efficientnet_b0,doctor,0.976744,0.884354,0.885118
1,protocol_A,RGB,efficientnet_b0,emergency,0.904762,0.884354,0.885118
2,protocol_A,RGB,efficientnet_b0,fire,0.909091,0.884354,0.885118
3,protocol_A,RGB,efficientnet_b0,help,0.975610,0.884354,0.885118
4,protocol_A,RGB,efficientnet_b0,robbery,0.894737,0.884354,0.885118
5,protocol_A,RGB,efficientnet_b0,sit_down,0.744186,0.884354,0.885118
6,protocol_A,RGB,efficientnet_b0,stand_up,0.790698,0.884354,0.885118
7,protocol_A,RGB,efficientnet_b0,doctor,0.976744,0.884354,0.885118
8,protocol_A,RGB,efficientnet_b0,emergency,0.904762,0.884354,0.885118
9,protocol_A,RGB,efficientnet_b0,fire,0.909091,0.884354,0.885118



========== RUNTIME SUMMARY ==========
{
    "num_total_experiments_collected": 80,
    "num_unique_protocols": 13,
    "num_unique_frame_sources": 2,
    "num_unique_models": 3,
    "total_train_time_minutes_sum": 6461.144535549482,
    "total_train_time_hours_sum": 107.68574225915803,
    "average_train_time_minutes": 87.31276399391191,
    "fastest_model_avg_videos_per_second": "mobilenet_v3_small",
    "best_model_avg_macro_f1": "efficientnet_b0"
}

All summary files saved in: /data/Sajjan_Singh/gesture_recognition/results/intermediate_report_bundle
